# Appliances Energy Prediction — Binary Classification
## CSE374 — Complete End-to-End Supervised Classification Study
### Models: Logistic Regression · KNN · Decision Tree · Neural Network (TensorFlow)
---

## Section 1 — Problem Framing and Data Understanding

**Problem:** We want to predict whether household appliance energy consumption is HIGH or LOW based on environmental sensor readings (temperature, humidity) collected every 10 minutes.

**Target Variable:** `Appliances` (Wh) → converted to binary target using median split:
- `0` = Low energy use (≤ 60 Wh)
- `1` = High energy use (> 60 Wh)

**Task Type:** Binary Classification

**Possible Challenges:**
- Slight class imbalance (~54% Low vs ~46% High) — handled with `class_weight='balanced'`
- Right-skewed appliance distribution — median split is more robust than mean
- Weak linear correlations in most features — may limit Logistic Regression
- Time-series nature of data — random splits may leak temporal patterns (acknowledged as limitation)
- Two random noise columns (rv1, rv2) — excluded from features

## Section 2 — Data Acquisition and Documentation

**Source:** UCI Machine Learning Repository — [Appliances Energy Prediction](https://archive.ics.uci.edu/dataset/374/electrical+grid+stability+simulated+data)

**Dataset Size:** ~19,735 rows × 29 columns

**Features:** 26 sensor features — temperature (T1–T9, T_out), humidity (RH_1–RH_9, RH_out), lights energy, and weather variables (Windspeed, Visibility, Tdewpoint, Press_mm_hg)

**Target:** `Appliances` (energy use in Wh)

**Modifications Made:**
1. Dropped `date` column (non-numeric)
2. Dropped `rv1`, `rv2` (random noise, zero correlation with target)
3. Converted `Appliances` to binary `target` using median (60 Wh) as threshold
4. Stratified 80/20 train/test split to preserve class ratios
5. StandardScaler fitted only on training set to avoid data leakage

## Section 3 — Imports

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,roc_auc_score, roc_curve, classification_report, ConfusionMatrixDisplay)

# ── TensorFlow / Keras (Shallow Neural Network) ──────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('Libraries loaded successfully!')
print(f'TensorFlow version: {tf.__version__}')

: 

## Section 4 — Data Cleaning and Preprocessing

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('energydata_complete.csv')

# Convert all columns to numeric (except 'date') — non-numeric values become NaN
for col in df.columns:
    if col != 'date':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with any missing values
df.dropna(inplace=True)

print(f'Shape      : {df.shape}')
print(f'Missing    : {df.isnull().sum().sum()}')
print(f'Duplicates : {df.duplicated().sum()}')
df.describe().round(2)

In [ ]:
# ── Engineer Binary Target ────────────────────────────────────────────────────
# Median of Appliances = 60 Wh
# 0 = Low energy (≤ 60 Wh)  |  1 = High energy (> 60 Wh)
df['target'] = (df['Appliances'] > df['Appliances'].median()).astype(int)

# Features: drop date (non-numeric), Appliances (source of target),
#           rv1/rv2 (random noise with zero predictive power)
X = df.drop(columns=['date', 'Appliances', 'rv1', 'rv2', 'target'])
y = df['target']

print('Features:', list(X.columns))
print('X shape :', X.shape)
print('\nClass distribution:')
print(y.value_counts())
print(f'Class balance: {y.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────────
# Stratified split ensures same class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# ── Feature Scaling ───────────────────────────────────────────────────────────
# IMPORTANT: Scaler is fit ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)        # transform only

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features         : {X_train.shape[1]}')

## Section 5 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Plot 1: Class Distribution ────────────────────────────────────────────────
counts = y.value_counts().sort_index()
plt.figure(figsize=(5, 5))
plt.bar(['Low (≤60 Wh)', 'High (>60 Wh)'], counts.values,
        color=['#1008F9', '#FF1807'], edgecolor='black', width=0.5)
for i, v in enumerate(counts.values):
    plt.text(i, v + 100, f'{v} ({v/len(y)*100:.1f}%)', ha='center', fontweight='bold')
plt.title('Fig 1: Class Distribution', fontweight='bold')
plt.ylabel('Count'); plt.ylim(0, 14000)
plt.tight_layout(); plt.show()

# Interpretation: There is a slight class imbalance — 54.4% Low vs 45.6% High.
# This is not severe, but we use class_weight='balanced' in Logistic Regression
# and Decision Tree, and stratified splits to ensure fair evaluation.

In [ ]:
# ── Plot 2: Appliances Energy Distribution with Threshold ─────────────────────
plt.figure(figsize=(8, 4))
plt.hist(df['Appliances'], bins=60, color='#90CAF9', edgecolor='black')
plt.axvline(df['Appliances'].median(), color='red', lw=2, linestyle='--',label=f'Median = {df["Appliances"].median():.0f} Wh')
plt.title('Fig 2: Appliances Energy Distribution (Classification Threshold)', fontweight='bold')
plt.xlabel('Energy Use (Wh)'); plt.ylabel('Frequency')
plt.legend(); plt.tight_layout(); plt.show()

# Interpretation: The distribution is strongly right-skewed — most readings
# are clustered between 50–150 Wh with a long tail to the right.
# Using the median (60 Wh) as the classification threshold is more robust
# than the mean because it is not affected by extreme values.

In [ ]:
# ── Plot 3: Mean Temperature Features by Class ────────────────────────────────
temp_cols = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T_out']
means = df.groupby('target')[temp_cols].mean()

x = np.arange(len(temp_cols)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w/2, means.loc[0], w, label='Low Energy', color='#2196F3', edgecolor='black')
ax.bar(x + w/2, means.loc[1], w, label='High Energy', color='#F44336', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(temp_cols)
ax.set_title('Fig 3: Mean Temperature by Class', fontweight='bold')
ax.set_ylabel('°C'); ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

# Interpretation: T6 (outside building area) shows the largest difference
# between classes. Indoor temperatures (T1–T5) differ only modestly.
# This suggests outdoor temperature may influence appliance use (e.g., heating/cooling).

In [ ]:
# ── Plot 4: Feature Correlation with Target ───────────────────────────────────
corr_with_target = X.corrwith(y).abs().sort_values(ascending=True)
colors = ['#F44336' if v > 0.1 else '#90CAF9' for v in corr_with_target]

plt.figure(figsize=(8, 7))
plt.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
plt.axvline(0.1, color='red', linestyle='--', lw=1, label='0.10 threshold')
plt.title('Fig 4: Feature Correlation with Target (|Pearson r|)', fontweight='bold')
plt.xlabel('|Pearson r|'); plt.legend()
plt.tight_layout(); plt.show()

# Interpretation: 'lights', T6, RH_6, and T_out show the strongest correlations
# with the target. Most features have weak linear correlation (< 0.10),
# which suggests that non-linear models (KNN, Decision Tree) may outperform
# Logistic Regression on this dataset.

In [ ]:
# ── Plot 5: Correlation Heatmap (Top Features) ────────────────────────────────
# Select the top 10 most correlated features for readability
top_features = corr_with_target.sort_values(ascending=False).head(10).index.tolist()
heatmap_data = X[top_features].copy()
heatmap_data['target'] = y

plt.figure(figsize=(10, 7))
sns.heatmap(heatmap_data.corr(), annot=True, fmt='.2f', cmap='coolwarm',linewidths=0.5, center=0)
plt.title('Fig 5: Correlation Heatmap (Top 10 Features + Target)', fontweight='bold')
plt.tight_layout(); plt.show()

# Interpretation: Several temperature features are highly correlated with each other
# (T1–T5 are all indoor rooms). This multicollinearity may hurt Logistic Regression
# coefficient interpretation but does not affect tree-based models or KNN significantly.

## Section 6 — Modeling

We train four classifiers. The test set is **never** used during model selection or hyperparameter tuning — only during final evaluation.

Cross-validation is performed on the training set to estimate generalization performance.

### 6.1 — Model 1: Logistic Regression

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────────
# class_weight='balanced' corrects for the slight class imbalance
# max_iter=1000 ensures the solver converges
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

# 5-Fold Cross-Validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc_lr = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='accuracy')
cv_f1_lr  = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='f1')
cv_auc_lr = cross_val_score(lr, X_train_sc, y_train, cv=cv, scoring='roc_auc')

print('=== Logistic Regression — 5-Fold CV (Training Set) ===')
print(f'  Accuracy : {cv_acc_lr.mean():.4f} ± {cv_acc_lr.std():.4f}')
print(f'  F1-Score : {cv_f1_lr.mean():.4f} ± {cv_f1_lr.std():.4f}')
print(f'  ROC-AUC  : {cv_auc_lr.mean():.4f} ± {cv_auc_lr.std():.4f}')

# Train final model on full training set
lr.fit(X_train_sc, y_train)
print('\nLogistic Regression trained successfully.')

### 6.2 — Model 2: K-Nearest Neighbors (KNN)

In [ ]:
# ── KNN — Find Best K Using Cross-Validation ──────────────────────────────────
# We test odd values of K from 1 to 21 to avoid ties in binary classification
k_values = list(range(1, 22, 2))  # [1, 3, 5, 7, ..., 21]
cv_scores_k = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # Using AUC as the selection metric
    score = cross_val_score(knn, X_train_sc, y_train, cv=cv, scoring='roc_auc').mean()
    cv_scores_k.append(score)

# Plot K vs AUC
plt.figure(figsize=(7, 4))
plt.plot(k_values, cv_scores_k, marker='o', color='#2196F3', linewidth=2)
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('CV ROC-AUC')
plt.title('KNN: Choosing Best K via Cross-Validation', fontweight='bold')
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Select best K
best_k = k_values[np.argmax(cv_scores_k)]
print(f'Best K = {best_k}  (CV AUC = {max(cv_scores_k):.4f})')

In [ ]:
# ── Train Final KNN with Best K ───────────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=best_k)

# Cross-validation metrics for reporting
cv_acc_knn = cross_val_score(knn, X_train_sc, y_train, cv=cv, scoring='accuracy')
cv_f1_knn  = cross_val_score(knn, X_train_sc, y_train, cv=cv, scoring='f1')
cv_auc_knn = cross_val_score(knn, X_train_sc, y_train, cv=cv, scoring='roc_auc')

print(f'=== KNN (K={best_k}) — 5-Fold CV (Training Set) ===')
print(f'  Accuracy : {cv_acc_knn.mean():.4f} ± {cv_acc_knn.std():.4f}')
print(f'  F1-Score : {cv_f1_knn.mean():.4f} ± {cv_f1_knn.std():.4f}')
print(f'  ROC-AUC  : {cv_auc_knn.mean():.4f} ± {cv_auc_knn.std():.4f}')

# Train final model on full training set
knn.fit(X_train_sc, y_train)
print(f'\nKNN (K={best_k}) trained successfully.')

### 6.3 — Model 3: Decision Tree

In [ ]:
# ── Decision Tree — Tune Max Depth Using Cross-Validation ────────────────────
# Too shallow → underfit, too deep → overfit
depths = list(range(2, 21))
cv_scores_dt = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, class_weight='balanced', random_state=42)
    score = cross_val_score(dt, X_train_sc, y_train, cv=cv, scoring='roc_auc').mean()
    cv_scores_dt.append(score)

# Plot depth vs AUC
plt.figure(figsize=(7, 4))
plt.plot(depths, cv_scores_dt, marker='s', color='#4CAF50', linewidth=2)
plt.xlabel('Max Tree Depth')
plt.ylabel('CV ROC-AUC')
plt.title('Decision Tree: Choosing Best Depth via Cross-Validation', fontweight='bold')
plt.xticks(depths)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(cv_scores_dt)]
print(f'Best max_depth = {best_depth}  (CV AUC = {max(cv_scores_dt):.4f})')

In [ ]:
# ── Train Final Decision Tree with Best Depth ─────────────────────────────────
dt = DecisionTreeClassifier(max_depth=best_depth, class_weight='balanced', random_state=42)

cv_acc_dt = cross_val_score(dt, X_train_sc, y_train, cv=cv, scoring='accuracy')
cv_f1_dt  = cross_val_score(dt, X_train_sc, y_train, cv=cv, scoring='f1')
cv_auc_dt = cross_val_score(dt, X_train_sc, y_train, cv=cv, scoring='roc_auc')

print(f'=== Decision Tree (depth={best_depth}) — 5-Fold CV (Training Set) ===')
print(f'  Accuracy : {cv_acc_dt.mean():.4f} ± {cv_acc_dt.std():.4f}')
print(f'  F1-Score : {cv_f1_dt.mean():.4f} ± {cv_f1_dt.std():.4f}')
print(f'  ROC-AUC  : {cv_auc_dt.mean():.4f} ± {cv_auc_dt.std():.4f}')

# Train final model on full training set
dt.fit(X_train_sc, y_train)
print(f'\nDecision Tree (depth={best_depth}) trained successfully.')

In [ ]:
# ── Visualize Top 3 Levels of the Decision Tree ───────────────────────────────
plt.figure(figsize=(18, 6))
plot_tree(dt, max_depth=3, feature_names=X.columns.tolist(),class_names=['Low', 'High'], filled=True, fontsize=8,impurity=False, proportion=True)
plt.title(f'Decision Tree (top 3 levels, full depth={best_depth})', fontweight='bold')
plt.tight_layout()
plt.show()

### 6.4 — Model 4: Shallow Neural Network (TensorFlow/Keras)

In [ ]:
# ── Build Shallow Neural Network ──────────────────────────────────────────────
# Architecture: Input → Dense(64, ReLU) → Dropout(0.3) → Dense(32, ReLU) → Dense(1, Sigmoid)
# This is 'shallow' — only 2 hidden layers (vs 'deep' networks with many more)

def build_nn(input_dim):
    """Build and compile the shallow neural network."""
    model = keras.Sequential([
        # Input layer
        layers.Input(shape=(input_dim,)),
        
        # First hidden layer: 64 neurons, ReLU activation
        layers.Dense(64, activation='relu'),
        
        # Dropout: randomly turn off 30% of neurons during training to prevent overfitting
        layers.Dropout(0.3),
        
        # Second hidden layer: 32 neurons, ReLU activation
        layers.Dense(32, activation='relu'),
        
        # Dropout for regularization
        layers.Dropout(0.2),
        
        # Output layer: 1 neuron, Sigmoid → outputs probability between 0 and 1
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compile: Adam optimizer, binary crossentropy loss (standard for binary classification)
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Build and show summary
nn_model = build_nn(input_dim=X_train_sc.shape[1])
nn_model.summary()

In [ ]:
# ── Train Neural Network ──────────────────────────────────────────────────────
# EarlyStopping stops training when validation loss stops improving
# This prevents overfitting and saves time
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,          # wait 10 epochs with no improvement before stopping
    restore_best_weights=True  # keep the best model weights
)

# Split a small validation set from training data (for monitoring during training)
# NOTE: We use X_train_sc (already scaled), and this val split is separate from X_test
X_tr, X_val, y_tr, y_val = train_test_split(X_train_sc, y_train, test_size=0.15, stratify=y_train, random_state=42)

history = nn_model.fit(X_tr, y_tr,validation_data=(X_val, y_val),epochs=100,verbose=1)

print(f'\nTraining stopped at epoch: {len(history.history["loss"])}')

In [ ]:
# ── Plot Training History ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss', color='#2196F3')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#F44336')
axes[0].set_title('Neural Network: Loss Curve', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='#2196F3')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='#F44336')
axes[1].set_title('Neural Network: Accuracy Curve', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Training History — Shallow Neural Network (TensorFlow)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Interpretation: If train and val curves track closely, the model generalizes well.
# A large gap between them would indicate overfitting.

## Section 7 — Performance Evaluation on Test Set

All four models are now evaluated on the **held-out test set** for the first time.

In [ ]:
# ── Helper Function: Evaluate Any Model ───────────────────────────────────────
def evaluate_model(name, y_true, y_pred, y_prob):
    """Compute and print accuracy, F1, AUC for a model."""
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    print(f'\n=== {name} — Test Set Results ===')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  F1-Score : {f1:.4f}')
    print(f'  ROC-AUC  : {auc:.4f}')
    print(f'\n{classification_report(y_true, y_pred, target_names=["Low", "High"])}')
    return acc, f1, auc

In [ ]:
# ── Get Predictions for All Models ────────────────────────────────────────────

# Logistic Regression
lr_pred  = lr.predict(X_test_sc)
lr_prob  = lr.predict_proba(X_test_sc)[:, 1]

# KNN
knn_pred = knn.predict(X_test_sc)
knn_prob = knn.predict_proba(X_test_sc)[:, 1]

# Decision Tree
dt_pred  = dt.predict(X_test_sc)
dt_prob  = dt.predict_proba(X_test_sc)[:, 1]

# Neural Network (returns probabilities directly, threshold at 0.5)
nn_prob  = nn_model.predict(X_test_sc).flatten()
nn_pred  = (nn_prob >= 0.5).astype(int)

# Evaluate all models
acc_lr,  f1_lr,  auc_lr  = evaluate_model('Logistic Regression', y_test, lr_pred,  lr_prob)
acc_knn, f1_knn, auc_knn = evaluate_model('KNN',                 y_test, knn_pred, knn_prob)
acc_dt,  f1_dt,  auc_dt  = evaluate_model('Decision Tree',       y_test, dt_pred,  dt_prob)
acc_nn,  f1_nn,  auc_nn  = evaluate_model('Neural Network',      y_test, nn_pred,  nn_prob)

In [ ]:
# ── Confusion Matrices — All 4 Models in One Figure ───────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

models_info = [
    ('Logistic Regression', y_test, lr_pred),
    (f'KNN (K={best_k})',   y_test, knn_pred),
    (f'Decision Tree (d={best_depth})', y_test, dt_pred),
    ('Neural Network',      y_test, nn_pred)
]

for ax, (name, y_true, y_pred) in zip(axes, models_info):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'High'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models (Test Set)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves — All 4 Models ─────────────────────────────────────────────────
plt.figure(figsize=(8, 6))

# Plot each model's ROC curve
for name, y_prob, color in [
    ('Logistic Regression', lr_prob,  '#1565C0'),
    (f'KNN (K={best_k})',   knn_prob, '#388E3C'),
    (f'Decision Tree (d={best_depth})', dt_prob, '#F57F17'),
    ('Neural Network',      nn_prob,  '#C62828')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC = {auc:.3f})')

# Diagonal line = random classifier
plt.plot([0,1], [0,1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.500)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models (Test Set)', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 8 — Model Comparison

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        f'KNN (K={best_k})',
        f'Decision Tree (depth={best_depth})',
        'Shallow Neural Network'
    ],
    'Test Accuracy': [f'{acc_lr:.4f}', f'{acc_knn:.4f}', f'{acc_dt:.4f}', f'{acc_nn:.4f}'],
    'Test F1-Score': [f'{f1_lr:.4f}',  f'{f1_knn:.4f}',  f'{f1_dt:.4f}',  f'{f1_nn:.4f}'],
    'Test ROC-AUC' : [f'{auc_lr:.4f}', f'{auc_knn:.4f}', f'{auc_dt:.4f}', f'{auc_nn:.4f}'],
    'Key Hyperparameters': [
        'C=1.0, class_weight=balanced',
        f'n_neighbors={best_k}',
        f'max_depth={best_depth}, class_weight=balanced',
        'Dense(64)→Dropout(0.3)→Dense(32), Adam, EarlyStopping'
    ]
})

print('='*80)
print('MODEL COMPARISON TABLE')
print('='*80)
print(comparison_df.to_string(index=False))
print('='*80)

In [ ]:
# ── Visual Comparison: Bar Chart ──────────────────────────────────────────────
model_names = ['Logistic\nRegression', f'KNN\n(K={best_k})',
               f'Decision\nTree (d={best_depth})', 'Neural\nNetwork']
accuracies = [acc_lr, acc_knn, acc_dt, acc_nn]
f1_scores  = [f1_lr,  f1_knn,  f1_dt,  f1_nn]
aucs       = [auc_lr, auc_knn, auc_dt, auc_nn]

x = np.arange(len(model_names))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - w,     accuracies, w, label='Accuracy', color='#1565C0', edgecolor='black')
bars2 = ax.bar(x,         f1_scores,  w, label='F1-Score', color='#388E3C', edgecolor='black')
bars3 = ax.bar(x + w,     aucs,       w, label='ROC-AUC',  color='#C62828', edgecolor='black')

# Add value labels on top of each bar
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(model_names)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy, F1-Score, ROC-AUC (Test Set)', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

### Model Comparison Analysis

The comparison table above shows all four models' performance on the test set. Based on the results:

- **Best Model Discussion:** The Neural Network and Decision Tree typically outperform Logistic Regression on this dataset because the feature-target relationships are non-linear. The correlation heatmap (EDA) confirmed that most features have weak linear correlation with the target, which inherently limits Logistic Regression's ceiling.

- **KNN** is competitive but computationally expensive at test time — it must compare each new point against all training samples.

- **Decision Tree** is interpretable (see the tree visualization above) and handles non-linear boundaries naturally through splits.

- **Neural Network** with Dropout and Early Stopping avoids overfitting and generally achieves the highest AUC, indicating the best separation between classes.

## Section 9 — Error Analysis

In [ ]:
# ── Identify Misclassified Samples (using best-performing model) ───────────────
# Use Neural Network predictions for error analysis
test_df = X_test.copy()
test_df['true_label']  = y_test.values
test_df['nn_pred']     = nn_pred
test_df['nn_prob']     = nn_prob
test_df['lr_pred']     = lr_pred
test_df['knn_pred']    = knn_pred
test_df['dt_pred']     = dt_pred

# Samples misclassified by ALL four models (hardest cases)
all_wrong = test_df[
    (test_df['true_label'] != test_df['nn_pred']) &
    (test_df['true_label'] != test_df['lr_pred']) &
    (test_df['true_label'] != test_df['knn_pred']) &
    (test_df['true_label'] != test_df['dt_pred'])
]

print(f'Total test samples              : {len(test_df)}')
print(f'NN misclassified                : {(test_df["true_label"] != test_df["nn_pred"]).sum()}')
print(f'LR misclassified                : {(test_df["true_label"] != test_df["lr_pred"]).sum()}')
print(f'KNN misclassified               : {(test_df["true_label"] != test_df["knn_pred"]).sum()}')
print(f'DT misclassified                : {(test_df["true_label"] != test_df["dt_pred"]).sum()}')

print(f'Misclassified Mutual by ALL 4   : {len(all_wrong)}')
print(f'\nClass distribution of hardest errors:')
print(all_wrong['true_label'].value_counts())